In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

from pathlib import Path
from decouple import AutoConfig

In [2]:
cwd = Path.cwd()
project_dir = cwd / "langchain" if (cwd / "langchain").is_dir() else cwd
config = AutoConfig(search_path=str(project_dir))
api_key = config("OPENAI_API_KEY")

In [3]:
openai = ChatOpenAI(api_key=api_key, model="gpt-4o-mini", temperature=0, store=False)

In [4]:
chain = (
    PromptTemplate.from_template(
        """
        Classifique a pergunta do usuário em exatamente uma destas categorias:
        - Assuntos Financeiros
        - Suporte Técnico
        - Atualização de Cadastro
        - Outras Informações

        Dúvidas sobre login, acesso à conta e redefinição de senha são Suporte Técnico.
        Retorne somente o nome exato da categoria, sem explicações.

        Pergunta: {query}
        Classificação:
        """
    )
    | openai
    | StrOutputParser() 
)

In [5]:
print(chain)

first=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\n        Classifique a pergunta do usuário em exatamente uma destas categorias:\n        - Assuntos Financeiros\n        - Suporte Técnico\n        - Atualização de Cadastro\n        - Outras Informações\n\n        Dúvidas sobre login, acesso à conta e redefinição de senha são Suporte Técnico.\n        Retorne somente o nome exato da categoria, sem explicações.\n\n        Pergunta: {query}\n        Classificação:\n        ') middle=[ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.7.1', 'langchain': '1.4.7', 'langchain-openai': '1.7.3'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False,

In [6]:
financial_chain = PromptTemplate.from_template(
    """
    Você é um especialista financeiro.
    Sempre responda às perguntas começando com 'Bem-vindo ao suporte financeiro!'.
    Responda à pergunta do usuário:
    Pergunta: {query}
    Resposta:
    """
) | openai
tech_support_chain = PromptTemplate.from_template(
    """
    Você é um especialista em suporte técnico.
    Sempre responda às perguntas começando com 'Bem-vindo ao suporte técnico!'.
    Ajude o usuário com seus problemas técnicos.
    Pergunta: {query}
    Resposta:
    """
) | openai
update_registration_chain = PromptTemplate.from_template(
    """
    Você é um representante de atendimento ao cliente.
    Sempre responda às perguntas começando com 'Bem-vindo ao suporte de cadastro'.
    Guie o usuário na atualização de suas informações de cadastro.
    Pergunta: {query}
    Resposta:
    """
) | openai
other_info_chain = PromptTemplate.from_template(
    """
    Você é um assistente de informações gerais.
    Sempre responda às perguntas começando com 'Bem-vindo ao suporte geral'.
    Forneça informações ao usuário sobre sua pergunta.
    Pergunta: {query}
    Resposta:
    """
) | openai

In [7]:
def route(info):
    topic = info["topic"].lower()
    if "financeiro" in topic:
        return financial_chain
    elif "técnico" in topic:
        return tech_support_chain
    elif "atualização" in topic or "cadastro" in topic:
        return update_registration_chain
    else:
        return other_info_chain

In [8]:
classification = chain.invoke({"query": "Como faço para redefinir minha senha?"})
print(classification)

Suporte Técnico


In [9]:
response_chain = route({"topic": classification})
print(response_chain)

first=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template="\n    Você é um especialista em suporte técnico.\n    Sempre responda às perguntas começando com 'Bem-vindo ao suporte técnico!'.\n    Ajude o usuário com seus problemas técnicos.\n    Pergunta: {query}\n    Resposta:\n    ") middle=[] last=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.7.1', 'langchain': '1.4.7', 'langchain-openai': '1.7.3'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'im

In [11]:
response = response_chain.invoke({"query": "Como faço para redefinir minha senha?"})
print(response)

content='Bem-vindo ao suporte técnico! Para redefinir sua senha, siga estas etapas:\n\n1. Acesse a página de login do serviço ou aplicativo.\n2. Clique na opção "Esqueci minha senha" ou "Redefinir senha".\n3. Insira o endereço de e-mail associado à sua conta e envie o formulário.\n4. Verifique seu e-mail para um link de redefinição de senha. Se não encontrar, verifique a pasta de spam ou lixo eletrônico.\n5. Clique no link e siga as instruções para criar uma nova senha.\n\nSe você tiver problemas durante o processo, não hesite em nos informar!' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 129, 'prompt_tokens': 62, 'total_tokens': 191, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provi

In [12]:
classification = chain.invoke({"query": "Como faço para pagar uma fatura atrasada?"})
response_chain = route({"topic": classification})
response = response_chain.invoke({"query": "Como faço para pagar uma fatura atrasada?"})
print(response)

content='Bem-vindo ao suporte financeiro! Para pagar uma fatura atrasada, siga estes passos:\n\n1. **Verifique o valor devido**: Consulte sua fatura para saber o montante total que precisa ser pago, incluindo possíveis juros ou taxas por atraso.\n\n2. **Escolha o método de pagamento**: Você pode pagar a fatura através de diferentes métodos, como:\n   - Online, pelo site ou aplicativo do seu banco ou da empresa credora.\n   - Por telefone, utilizando o serviço de atendimento ao cliente.\n   - Pessoalmente, em uma agência ou ponto de atendimento autorizado.\n   - Por meio de correio, enviando um cheque ou ordem de pagamento.\n\n3. **Realize o pagamento**: Siga as instruções do método escolhido para efetuar o pagamento. Se estiver pagando online, tenha em mãos seus dados bancários ou informações do cartão de crédito.\n\n4. **Confirme o pagamento**: Após realizar o pagamento, verifique se ele foi processado corretamente. Guarde o comprovante de pagamento para futuras referências.\n\n5. **E

In [13]:
classification = chain.invoke({"query": "Quero alterar meu endereço de e-mail."})
response_chain = route({"topic": classification})
response = response_chain.invoke({"query": "Quero alterar meu endereço de e-mail."})
print(response)

content='Bem-vindo ao suporte de cadastro! Para alterar seu endereço de e-mail, siga estas etapas:\n\n1. Acesse sua conta no nosso site.\n2. Vá até a seção de "Configurações" ou "Meu Perfil".\n3. Encontre a opção "Alterar E-mail" ou "Atualizar E-mail".\n4. Insira seu novo endereço de e-mail e confirme-o.\n5. Você pode receber um e-mail de confirmação no novo endereço. Clique no link de confirmação para validar a alteração.\n\nSe precisar de mais assistência, estou aqui para ajudar!' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 115, 'prompt_tokens': 66, 'total_tokens': 181, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system

In [14]:
classification = chain.invoke({"query": "Qual a missão da empresa?"})
response_chain = route({"topic": classification})
response = response_chain.invoke({"query": "Qual a missão da empresa?"})
print(response)

content='Bem-vindo ao suporte geral. A missão de uma empresa geralmente descreve seu propósito fundamental e os objetivos que busca alcançar. Ela pode variar de acordo com o setor e a visão dos fundadores, mas, em geral, a missão reflete os valores centrais da organização e como ela pretende impactar seus clientes, colaboradores e a sociedade. Para obter informações específicas sobre a missão de uma empresa em particular, recomendo visitar o site oficial da empresa ou consultar seus materiais institucionais. Se precisar de mais detalhes, estou à disposição!' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 108, 'prompt_tokens': 61, 'total_tokens': 169, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}